# Cleanup and Installation
Run these commands to clean up previous data and install required CLI tools for the RGB sandbox demo.

# Cleanup and Installation
Run these commands to clean up previous data and install required CLI tools for the RGB sandbox demo.

In [ ]:
# Remove data directories and generated files
rm -fr data{0,1} wallets consignment.yaml contracts/usdt.yaml

# Remove installed crates
rm -r bp-wallet rgb-cmd

# Install bp-wallet CLI
cargo install bp-wallet --version 0.11.1-alpha.2 --root ./bp-wallet --features=cli,hot

# Install rgb-cmd CLI
cargo install rgb-cmd --version 0.11.1-rc.5 --root ./rgb-cmd


    Updating crates.io index
  Installing bp-wallet v0.11.1-alpha.2+unreviewedomplete; 0 pending        
    Updating crates.io index
     Locking 244 packages to latest compatible versionsete; 1 pending      
      Adding amplify v4.8.1 (available: v4.9.0)
      Adding strict_encoding v2.8.2 (available: v2.9.1)
      Adding strict_types v2.8.3 (available: v2.9.1)
      Adding toml v0.8.23 (available: v0.9.8)
      Adding vesper-lang v0.2.1 (available: v0.2.3)
   Compiling libc v0.2.177==============>   ] 220 complete; 1 pending      
   Compiling proc-macro2 v1.0.103
   Compiling quote v1.0.42
   Compiling unicode-ident v1.0.22
   Compiling serde_core v1.0.228
   Compiling cfg-if v1.0.4
   Compiling serde v1.0.228
   Compiling typenum v1.19.0
   Compiling version_check v0.9.5     ] 0/217: proc-macro2(build.rs), ty...
   Compiling syn v1.0.109             ] 1/217: proc-macro2(build.rs), ve...
   Compiling generic-array v0.14.9    ] 2/217: proc-macro2(build.rs), ve...
   Compiling jobse

: 101

In [4]:
docker-compose down -v && docker-compose up -d

just setup-bitcoind


[+] Running 0/5
 ⠋ Container electrs                Stopping                               0.1s 
 ⠋ Container polar-n1-backend       Stoppi...                              0.1s 
 ⠋ Container mempool_frontend_btc   St...                                  0.1s 
 ⠋ Container rgb-sandbox-esplora-1  S...                                   0.1s 
 ⠋ Container mempool_backend_btc    Sto...                                 0.1s 
 ⠋ Container electrs                Stopping                               0.1s 
 ⠋ Container polar-n1-backend       Stoppi...                              0.1s 
 ⠋ Container mempool_frontend_btc   St...                                  0.1s 
 ⠋ Container rgb-sandbox-esplora-1  S...                                   0.1s 
 ⠋ Container mempool_backend_btc    Sto...                                 0.1s 
[+] Running 0/5
 ⠙ Container electrs                Stopping                               0.2s 
 ⠙ Container polar-n1-backend       Stoppi...                              0.

In [54]:


echo '{"jsonrpc": "2.0", "method": "blockchain.block.header", "params": [107], "id": 0}' | netcat -w1 localhost 50001


{"error":{"code":-32603,"message":"missing header"},"id":0,"jsonrpc":"2.0"}




# RGB Asset Minting and Transfer (Manual Demo)
This notebook contains the step-by-step commands for minting and transferring an RGB asset, following the manual demo instructions. The Bitcoin infrastructure is assumed to be already initialized and running.


## 1. Set up aliases and environment variables
These commands make it easier to use the RGB and wallet CLIs.


In [33]:
# Aliases for easier CLI usage
alias bp="bp-wallet/bin/bp"
alias bphot="bp-wallet/bin/bp-hot"
alias rgb0="rgb-cmd/bin/rgb -n regtest --electrum=localhost:50001 -d data0 -w issuer"
alias rgb1="rgb-cmd/bin/rgb -n regtest --electrum=localhost:50001 -d data1 -w rcpt1"
alias bcli='docker compose exec -T bitcoind bitcoin-cli -regtest -rpcuser=polaruser -rpcpassword=polarpass'

# Environment variables
CLOSING_METHOD="opret1st"
CONSIGNMENT="consignment.rgb"
PSBT="tx.psbt"
SCHEMATA_DIR="rgb-schemas/schemata"
WALLET_PATH="wallets"
KEYCHAIN="<0;1;9>"


## 2. Prepare Bitcoin wallets
Assumes Bitcoin Core wallet is already created and funded. Remove old wallets and create new ones for issuer and receiver.

In [30]:
# Remove old wallets if needed
rm -fr $WALLET_PATH

# Create wallet directory
mkdir -p $WALLET_PATH

# Seed password definition
export SEED_PASSWORD="seed test password"

# Issuer wallet: generate seed and derive account
bphot seed "$WALLET_PATH/0.seed" > issuer_seed_out.txt
bphot derive -N -s bip86 "$WALLET_PATH/0.seed" "$WALLET_PATH/0.derive" > issuer_derive_out.txt
account_0=$(grep -oE '\[[0-9a-f]{8}/86h/1h/0h\][^ ]+' issuer_derive_out.txt | head -n1)
descriptor_0="$account_0/$KEYCHAIN/*"

# Receiver wallet: generate seed and derive account
bphot seed "$WALLET_PATH/1.seed" > receiver_seed_out.txt
bphot derive -N -s bip86 "$WALLET_PATH/1.seed" "$WALLET_PATH/1.derive" > receiver_derive_out.txt
account_1=$(grep -oE '\[[0-9a-f]{8}/86h/1h/0h\][^ ]+' receiver_derive_out.txt | head -n1)
descriptor_1="$account_1/$KEYCHAIN/*"

echo "account_0=\"$account_0\""
echo "descriptor_0=\"$descriptor_0\""
echo "account_1=\"$account_1\""
echo "descriptor_1=\"$descriptor_1\""


BP: command-line tool for working with seeds and private keys in bitcoin protocol
    by LNP/BP Standards Association

    by LNP/BP Standards Association

BP: command-line tool for working with seeds and private keys in bitcoin protocol
    by LNP/BP Standards Association

BP: command-line tool for working with seeds and private keys in bitcoin protocol
    by LNP/BP Standards Association

BP: command-line tool for working with seeds and private keys in bitcoin protocol
    by LNP/BP Standards Association

BP: command-line tool for working with seeds and private keys in bitcoin protocol
    by LNP/BP Standards Association

BP: command-line tool for working with seeds and private keys in bitcoin protocol
    by LNP/BP Standards Association

BP: command-line tool for working with seeds and private keys in bitcoin protocol
    by LNP/BP Standards Association

account_0="[0daf97d9/86h/1h/0h]tpubDCAN2Q3SZBm8q7JV9LNFsc8D9TfGsZcrAZKrwgWZ2YcLphCCX3dnRn3Ub2ZyiDx64KYaMMqARH68tXfRYbbEem9kzFTkTFx

## 3. Set up RGB wallets
Create RGB wallets for issuer and receiver, and import the NIA schema.

In [34]:

# Issuer RGB wallet
rgb0 create --wpkh $descriptor_0 issuer

# Receiver RGB wallet
rgb1 create --wpkh $descriptor_1 rcpt1

# Import NIA schema into both wallets
rgb0 import $SCHEMATA_DIR/NonInflatableAsset.rgb
rgb1 import $SCHEMATA_DIR/NonInflatableAsset.rgb

# Get schema ID
schema_id=$(rgb0 schemata | grep NonInflatableAsset | awk '{print $2}')


Loading descriptor from command-line argument
SyncingSyncing keychain 0 .......... keychain 1 .......... keychain 9 .......... success
Saving the wallet as 'issuer' ... success

SyncingSyncing keychain 0 .......... keychain 1 .......... keychain 9 .......... success
Saving the wallet as 'issuer' ... success

Loading descriptor from command-line argument
Loading descriptor from command-line argument
SyncingSyncing keychain 0 .......... keychain 1 .......... keychain 9 .......... success
Saving the wallet as 'rcpt1' ... success

SyncingSyncing keychain 0 .......... keychain 1 .......... keychain 9 .......... success
Saving the wallet as 'rcpt1' ... success

Importing kit rgb:kit:qxyONQWD-WY7Sha7-wzsRSMW-MaNI6PI-T74uzx9-sBzUztg:
- schema NonInflatableAsset RWhwUfTMpuP2Zfx1~j4nswCANGeJrYOqDcKelaMV4zU#remote-digital-pegasus
- script library alu:q~CZ0ovt-UN9eBlc-VMn86mz-Kfd3ywu-f7~9jTB-k6A8tiY#japan-nylon-center
- strict types: 35 definitions
Importing kit rgb:kit:qxyONQWD-WY7Sha7-wzsRSMW-Ma

## 4. Prepare UTXOs
Generate addresses, fund wallets, and gather outpoints for asset issuance and receiving.

In [ ]:
# Generate addresses for asset issuance and receiving
addr_issue=$(rgb0 address -k 9 | grep '&9/0' | awk '{print $2}')
if [ -z "$addr_issue" ]; then
  addr_issue=$(rgb0 address -k 9 | grep -Eo 'bcrt1[qpz0-9a-z]+' | head -n1)
fi
addr_receive=$(rgb1 address -k 9 | grep '&9/0' | awk '{print $2}')
if [ -z "$addr_receive" ]; then
  addr_receive=$(rgb1 address -k 9 | grep -Eo 'bcrt1[qpz0-9a-z]+' | head -n1)
fi
echo "addr_issue=$addr_issue"
echo "addr_receive=$addr_receive"
# Fund wallets (assumes bcli and wallet are ready)
bcli -rpcwallet=default sendtoaddress "$addr_issue" 1
bcli -rpcwallet=default sendtoaddress "$addr_receive" 1
bcli -rpcwallet=default -generate 1



Loading descriptor from wallet issuer ... success
Loading descriptor from wallet issuer ... success
Loading descriptor from wallet issuer ... success
Loading descriptor from wallet rcpt1 ... success
Loading descriptor from wallet rcpt1 ... success
Loading descriptor from wallet rcpt1 ... success
Loading descriptor from wallet rcpt1 ... success
addr_issue=bcrt1q64xkdu5kdnpzhmwlau453m687tqj08atf2sl9l
addr_issue=bcrt1q64xkdu5kdnpzhmwlau453m687tqj08atf2sl9l
addr_receive=bcrt1qlx2tj45cgmaavw9ptqw3jpnumdn0qmgmqsfsuv
addr_receive=bcrt1qlx2tj45cgmaavw9ptqw3jpnumdn0qmgmqsfsuv
2c2633965465cea270efa781787bb27129117ce82db13cbabd29c72f5dd92b9b
2c2633965465cea270efa781787bb27129117ce82db13cbabd29c72f5dd92b9b
f5f44d14edba7f61c99a53a014be74779fd9f3597a5460c20f823236feb734dc
f5f44d14edba7f61c99a53a014be74779fd9f3597a5460c20f823236feb734dc
{
  "address": "bcrt1q7z59d8my8fwnrpepgckuyav4y54mzwzemg688v",
  "blocks": [
    "1c3523f63465913139f4e57103ab0e3742f3afecfbbae2a5b4fe10f6d89c8a67"
  ]
}
{
  "address

In [ ]:
# Sync wallets and gather outpoints
rgb0 utxos --sync > issuer_utxos.txt
outpoint_issue=$(grep -Eo '[0-9a-f]{64}:[0-9]+' issuer_utxos.txt | head -n1)
rgb1 utxos --sync > rcpt_utxos.txt
outpoint_receive=$(grep -Eo '[0-9a-f]{64}:[0-9]+' rcpt_utxos.txt | head -n1)
echo "outpoint_issue=$outpoint_issue"
echo "outpoint_receive=$outpoint_receive"


Loading descriptor from wallet issuer ... success
Syncing keychain 0 .......... keychain 1 .......... keychain 9 ..................... success
Loading descriptor from wallet issuer ... success
Syncing keychain 0 .......... keychain 1 .......... keychain 9 ..................... success
Loading descriptor from wallet issuer ... success
Loading descriptor from wallet rcpt1 ... success
Loading descriptor from wallet rcpt1 ... success
Syncing keychain 0 .......... keychain 1 .......... keychain 9 ................... success
Loading descriptor from wallet rcpt1 ... success
Syncing keychain 0 .......... keychain 1 .......... keychain 9 ................... success
Loading descriptor from wallet rcpt1 ... success
outpoint_issue=2c2633965465cea270efa781787bb27129117ce82db13cbabd29c72f5dd92b9b:0
outpoint_issue=2c2633965465cea270efa781787bb27129117ce82db13cbabd29c72f5dd92b9b:0
outpoint_receive=b89dd88220af591e7e80116863d4e5e2293c9e3575986de7100b850d830b9e85:0
outpoint_receive=b89dd88220af591e7e801

## 5. Asset issuance
Prepare the contract file and issue the asset using the RGB CLI.

In [79]:
# Prepare contract file for asset issuance
sed \
  -e "s/schema_id/$schema_id/" \
  -e "s/issued_supply/1000/" \
  -e "s/txid:vout/$outpoint_issue/" \
  contracts/usdt.yaml.template > contracts/usdt.yaml

# Issue the asset
rgb0 issue "ssi:issuer" contracts/usdt.yaml

# Get contract ID
contract_id=$(rgb0 contracts | grep NonInflatableAsset | awk '{print $1}')

echo $contract_id


A new contract rgb:R8FMZy9m-iAs~MeW-Lq8UF_q-Hob0OAv-GIdDUtE-Cjf9zIY is issued and added to the stash.
Use `export` command to export the contract.

Use `export` command to export the contract.





Now we set up the contract id on the following field

In [80]:
contract_id=rgb:ToP41bKw-ECoREqB-tuDQlf1-NWh6rt6-0Qw4pEf-zGNJUb4


## 6. Transfer: Receiver generates invoice
The receiver generates an invoice to receive assets.

In [91]:
# Receiver generates invoice for 100 units
invoice=$(rgb1 invoice --amount 100 "$contract_id")
echo $invoice


Loading descriptor from wallet rcpt1 ... success
rgb:ToP41bKw-ECoREqB-tuDQlf1-NWh6rt6-0Qw4pEf-zGNJUb4/~/BF/bcrt:utxob:yk3F6zcz-lI2zTBx-85NXExZ-weFEn60-gTmmAv~-BZxb513-QHBzT?assignment_name=assetOwner
rgb:ToP41bKw-ECoREqB-tuDQlf1-NWh6rt6-0Qw4pEf-zGNJUb4/~/BF/bcrt:utxob:yk3F6zcz-lI2zTBx-85NXExZ-weFEn60-gTmmAv~-BZxb513-QHBzT?assignment_name=assetOwner


## 7. Transfer: Sender initiates asset transfer
Sender creates the consignment and PSBT for the transfer.

In [92]:
# Sender creates consignment and PSBT
rgb0 transfer "$invoice" "data0/$CONSIGNMENT" "data0/$PSBT"

# (Optional) Inspect consignment
rgb0 inspect "data0/$CONSIGNMENT" > consignment.yaml

# Exchange consignment file to receiver
cp data0/$CONSIGNMENT data1/$CONSIGNMENT


Loading descriptor from wallet issuer ... success




## 8. Receiver: Validate transfer
Receiver validates the consignment before accepting.

In [93]:
# Receiver validates consignment
rgb1 validate "data1/$CONSIGNMENT"


The provided consignment is valid




## 9. Sender: Sign and broadcast transaction
Sender signs, finalizes, and broadcasts the transaction after receiver validation.

In [94]:
# Sender signs the PSBT
bphot sign -N "data0/$PSBT" "$WALLET_PATH/0.derive"

# Finalize and broadcast transaction
rgb0 finalize -p data0/$PSBT data0/${PSBT%psbt}tx


BP: command-line tool for working with seeds and private keys in bitcoin protocol
    by LNP/BP Standards Association

Signing data0/tx.psbt with wallets/0.derive
    by LNP/BP Standards Association

Signing data0/tx.psbt with wallets/0.derive
Signing key: [0daf97d9/86h/1h/0h]tpubDCAN2Q3SZBm8q7JV9LNFsc8D9TfGsZcrAZKrwgWZ2YcLphCCX3dnRn3Ub2ZyiDx64KYaMMqARH68tXfRYbbEem9kzFTkTFxk5DBCrK2EtB9
Signing using testnet signer
PSBT version: v0
Transaction id: e6c854809d73c8311dad942b56d35649e55eca8bc40babc30e6cee6c3a2a7a9d
Done 1 signatures, saved to data0/tx.psbt
Signing key: [0daf97d9/86h/1h/0h]tpubDCAN2Q3SZBm8q7JV9LNFsc8D9TfGsZcrAZKrwgWZ2YcLphCCX3dnRn3Ub2ZyiDx64KYaMMqARH68tXfRYbbEem9kzFTkTFxk5DBCrK2EtB9
Signing using testnet signer
PSBT version: v0
Transaction id: e6c854809d73c8311dad942b56d35649e55eca8bc40babc30e6cee6c3a2a7a9d
Done 1 signatures, saved to data0/tx.psbt




cHNidP8BAH0CAAAAAaFG/3xIwDNUHS0zOfvhY0Q7JgXmm1G/6Vt95qruPfe6AQAAAAAAAAAAAgAAAAAAAAAAImogDTbuOvTK3lYvzcWar8hjVlOouXLaRCvWqhRH

## 10. Confirm transaction and sync wallets
Confirm the transaction and update wallet states.

In [95]:
# Confirm transaction (mine a block)
bcli -rpcwallet=default -generate 1

# Sync wallets
rgb0 utxos --sync
rgb1 utxos --sync


{
  "address": "bcrt1qc0xhpfcnnyygqktvfjp88m36nxuju4y2ppdzrk",
  "blocks": [
    "24903f4ca5c1b49e7014ab13ed235ea67ce98a930b26a5256648764da3718c50"
  ]
}
  "address": "bcrt1qc0xhpfcnnyygqktvfjp88m36nxuju4y2ppdzrk",
  "blocks": [
    "24903f4ca5c1b49e7014ab13ed235ea67ce98a930b26a5256648764da3718c50"
  ]
}
Loading descriptor from wallet issuer ... success
Loading descriptor from wallet issuer ... success
Syncing keychain 0 .......... keychain 1 .......... keychain 9 ........................ success
Balance of wpkh([0daf97d9/86h/1h/0h]tpubDCAN2Q3SZBm8q7JV9LNFsc8D9TfGsZcrAZKrwgWZ2YcLphCCX3dnRn3Ub2ZyiDx64KYaMMqARH68tXfRYbbEem9kzFTkTFxk5DBCrK2EtB9/<0;1;9>/*)

Height	   Amount, ṩ	Outpoint                                                            
bcrt1qnlxm9gv9qe7u9mv3mgld2zxec5uulxqqwtpllj	&9/8
105	   100000000	553797c8318c9ed8288de23b1243427f44d1c2c29d2af12d1bc70a03d94644eb:1

bcrt1q73625knnwttz6yzt5qn6sxp4uexlfg40g0fy4z	&9/13
109	    99998800	e6c854809d73c8311dad942b56d35649e55eca8bc40bab

## 11. Receiver: Accept transfer
Receiver accepts the transfer and updates contract state.

In [96]:
# Receiver accepts the transfer
rgb1 accept "data1/$CONSIGNMENT"

# Show updated contract state (receiver)
rgb1 state "$contract_id"

# Show updated contract state (issuer)
rgb0 state "$contract_id"


Transfer accepted into the stash


Loading descriptor from wallet rcpt1 ... success

Global:
  spec := ticker "USDT", name "USD Tether", details ~, precision indivisible
  terms := text "demo NIA asset", media ~
  issuedSupply := 1000

Owned:
  State      	Seal                                                                          	Witness
  assetOwner:
          100	b89dd88220af591e7e80116863d4e5e2293c9e3575986de7100b850d830b9e85:0	b32a23d91c6f383012c240fb0148f88c5ecb97519e75a51e331e30ae31e02328 (tentative) 
          100	b89dd88220af591e7e80116863d4e5e2293c9e3575986de7100b850d830b9e85:0	e6c854809d73c8311dad942b56d35649e55eca8bc40babc30e6cee6c3a2a7a9d (tentative) 
          100	b89dd88220af591e7e80116863d4e5e2293c9e3575986de7100b850d830b9e85:0	baf73deeaae67d5be9bf519be605263b4463e1fb39332d1d5433c0487cff46a1 (tentative) 

Loading descriptor from wallet rcpt1 ... success

Global:
  spec := ticker "USDT", name "USD Tether", details ~, precision indivisible
  terms := text "demo NIA ass